# 🎯 [PRC07_M03] 공급선 단일화 및 납기 지연 이력 기반 1차 조달 위험 스크리닝 알고리즘 구현
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/PRC07/prc07_m03_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 조달·계약·국산화 (Level 3) |
| **미션 ID** | PRC07_M03 |
| **미션 주제** | 공급선 단일화 및 납기 지연 이력 기반 1차 조달 위험 스크리닝 알고리즘 구현 |
| **소요 시간** | 약 2.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
무기체계 부품은 해외 의존도가 높고 공급선이 하나뿐이거나 단종(DMSMS) 공고가 나오면 조달이 막히기 쉬운데, 위험 신호를 사람이 문서와 전화로 모으면 국산화 후보를 찾기까지 수개월이 걸린다. 본 프로젝트는 부품 공급망 자료에서 조달 병목과 위험 신호를 찾고 국산화 대체 후보를 자동으로 추천하는 도구를 만드는 과제로, 업무 단계별 병목과 위험 신호 기준, 추천 수정안, 계약 의뢰 근거와 변경 이력, 업무 효과 측정까지 한 흐름으로 구현한다. 교육용 가상 부품 100종과 가상 국가·가상 업체 자료로 공급망 데이터 스키마 정의, 공급업체 자료 정제와 단종 공고 파싱, 룰 기반 조달 위험 스크리닝, 텍스트 유사도와 지식 그래프를 쓴 대체 부품 추천, 붕괴 위험도와 조기경보, 위험 지도 대시보드와 오류 처리, 규격 검토와 승인 HITL 워크플로우, 공급망 차단 스트레스 시험, AAR 가중치 재보정, 통합 패키지까지 단계별로 만든다. 최종 목표는 위험 부품을 조기에 찾아 국산화 후보를 근거와 함께 제시하되, 규격 적합성 판단과 조달 승인은 담당자가 하도록 지원하는 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

> **💡 [오프라인 폐쇄망 및 로컬 주피터 환경 안내]**
> - 외부 인터넷망(Colab) 외에 군 인트라넷 또는 로컬 환경에서 실행할 경우, 아래 명령어로 필요한 표준 라이브러리를 사전 설치하십시오:
>   `pip install numpy pandas matplotlib scipy scikit-learn`
> - 인터넷이 제한된 환경의 경우 사전 번들링된 Wheel(`.whl`) 패키지 디렉토리를 통해 설치할 수 있습니다:
>   `pip install --no-index --find-links=./wheels/ -r requirements.txt`

> 📁 **[안내: 프로젝트 표준 산출물 디렉터리 가이드]**
> - 본 프로젝트는 단순한 일회성 코드 실습을 넘어 실제 부대에 배포 가능한 업무 도구(Tool)를 완성하는 과정입니다.
> - 실습 중 작성하는 결과물은 역할에 따라 아래 표준 디렉터리에 자동으로 저장·누적됩니다:
>   - `docs/`: 조달 프로세스 분석서, 스키마 정의서, UAT 보고서, 사용자 매뉴얼 등 행정·기술 문서
>   - `src/`: 데이터 로더, 16종 위험 스크리닝 엔진, AI 추천 모델, 대시보드 UI, HITL 결재 FSM 등 핵심 실행 모듈
>   - `dist/`: 폐쇄망 및 타 부서 배포를 위한 원클릭 설치 스크립트(`setup.sh`) 및 최종 패키지

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")

# ============================================================================
# [산출물 자동 저장 유틸리티] 프로젝트 표준 디렉토리(docs/, src/, dist/) 자동 연계
# ============================================================================
def save_deliverable(filepath: str, content: str):
    """프로젝트 표준 deliverables 디렉토리에 산출물 파일을 자동 생성합니다."""
    dirpath = os.path.dirname(filepath)
    if dirpath:
        os.makedirs(dirpath, exist_ok=True)
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(content.strip() + '\n')
    print(f"📦 [산출물 저장 완료] {filepath}")


---

## 🎯 미션 03. 공급선 단일화 및 납기 지연 이력 기반 1차 조달 위험 스크리닝 알고리즘 구현
**소요시간**: 약 2.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

공급선 단일화 및 납기 지연 이력 기반 조달 위험 스크리닝 알고리즘을 구현한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **공급선 단일화와 납기 지연 이력 등을 점검하는 조달 위험 룰 15종 이상을 코딩해 부품별 1차 스크리닝 등급을 산출한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **단일 공급선, 해외 의존, 리드타임 초과, 납기 지연률, 단종 예고·확정, 최종구매 임박, 단가 급등, 재고 가용기간 부족 등 16종 룰을 YAML 설정으로 두고 가중치 합으로 고위험·주의·관심 등급을 낸다 과정을 진행하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/engine/prc07_rule_engine.py` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군 규정 및 지침 조항과의 정확한 일치성

#### 미션 PASS 조건

- 정량 기준: 업무 검증 룰 15종 이상 코딩, 룰 처리 지연시간 50ms 이하


- 진행 기준: 룰 엔진 단위 테스트 100% 통과


- 검증 방법: 경계 조건(Boundary condition)에 대한 룰 판정 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""공급선 단일화·납기 지연 이력 등 16종 조달 위험 룰(YAML 설정)을 코딩해 부품별 1차 스크리닝 등급을 산출하고, 룰 처리 지연시간·과거 단절 이력 대비 탐지력·룰별 적중 현황·규정 변경 시 설정만 바꾸는 확장성을 확인한다."""
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 무기체계 부품·공급망 데이터 생성기 (실제 체계·업체·국가·단가 정보 없음)
# 부품 100종(해외 조달 약 80%)의 공급망 속성과, 국내 부품 카탈로그(과거 승인된 국산 대체품 +
# 설명은 비슷하지만 규격이 맞지 않는 유사 후보)를 함께 만든다.
# ---------------------------------------------------------------------------
COUNTRIES = ["가상국가A", "가상국가B", "가상국가C", "국내"]
CATS = {
    "전원모듈": dict(words=["DC-DC", "컨버터", "절연형", "전원", "정출력"], syn={"컨버터": "변환장치", "절연형": "아이솔레이션", "정출력": "안정화"}, std="MIL-STD-704", iface="단자대"),
    "커넥터": dict(words=["원형", "커넥터", "방수형", "다핀", "나사체결"], syn={"커넥터": "접속자", "방수형": "밀폐형", "나사체결": "스크루결합"}, std="MIL-DTL-38999", iface="원형핀"),
    "유압밸브": dict(words=["유압", "솔레노이드", "밸브", "방향제어", "4포트"], syn={"밸브": "개폐장치", "방향제어": "유로전환", "솔레노이드": "전자식"}, std="SAE-AS5440", iface="플랜지"),
    "베어링": dict(words=["볼", "베어링", "고속", "밀봉형", "회전축"], syn={"베어링": "축받이", "밀봉형": "실드형", "고속": "고회전"}, std="ISO-492", iface="축결합"),
    "광학렌즈": dict(words=["적외선", "렌즈", "게르마늄", "반사방지", "집광"], syn={"렌즈": "광학창", "집광": "초점형성", "반사방지": "AR코팅"}, std="MIL-PRF-13830", iface="마운트"),
    "자이로센서": dict(words=["광섬유", "자이로", "각속도", "센서", "관성"], syn={"센서": "감지기", "각속도": "회전율", "관성": "INS용"}, std="IEEE-952", iface="직렬통신"),
    "RF증폭기": dict(words=["RF", "전력증폭기", "광대역", "모듈", "송신"], syn={"전력증폭기": "파워앰프", "송신": "출력단", "광대역": "와이드밴드"}, std="MIL-STD-461", iface="SMA"),
    "제어보드": dict(words=["FPGA", "제어보드", "임베디드", "연산", "입출력"], syn={"제어보드": "컨트롤카드", "연산": "신호처리", "임베디드": "내장형"}, std="VITA-46", iface="백플레인"),
}
V_OPTS, W_OPTS = [12, 24, 28], [25, 50, 100]
VENDORS = ["가상정밀(주)", "가상전자(주)", "가상유공압(주)", "가상광학(주)", "가상테크(주)", "가상시스템(주)"]


def describe(cat, words, s):
    return f"{cat} {' '.join(words)} {s['v']}V {s['w']}W {s['tmin']}~{s['tmax']}C {s['std']}"


def _sig(x):
    return 1 / (1 + np.exp(-x))


def make_parts(n=100, seed=7):
    """부품 목록(공급망 속성 포함)과 국내 카탈로그를 만든다. 반환: (parts DataFrame, catalog DataFrame)"""
    rng = np.random.default_rng(seed)
    cats, rows, cat_rows = list(CATS), [], []
    for i in range(n):
        cat = cats[i % len(cats)]; c = CATS[cat]
        spec = {"v": int(rng.choice(V_OPTS)), "w": int(rng.choice(W_OPTS)), "tmin": int(rng.choice([-55, -40])),
                "tmax": int(rng.choice([85, 125])), "size_mm": float(round(rng.uniform(20, 120), 1)), "std": c["std"]}
        words = [str(x) for x in rng.permutation(c["words"])[:4]]
        country = str(rng.choice(COUNTRIES, p=[0.35, 0.3, 0.15, 0.2]))
        overseas = country != "국내"
        n_sup = int(rng.choice([1, 2, 3], p=[0.45, 0.35, 0.2])) if overseas else int(rng.choice([1, 2, 3]))
        lead = float(round(rng.gamma(3, 45 if overseas else 25) + 20))
        eol = str(rng.choice(["정상", "단종예고", "단종"], p=[0.7, 0.2, 0.1]))
        crit = str(rng.choice(["A", "B", "C"], p=[0.3, 0.4, 0.3]))
        z = (-2.6 + 1.1 * (n_sup == 1) + 0.9 * overseas + 0.006 * (lead - 100) + 1.2 * (eol == "단종예고")
             + 2.0 * (eol == "단종") + rng.normal(0, 0.6))
        p_late = float(np.clip(_sig(z) * 0.8 + 0.05, 0, 0.9))
        late = int(rng.binomial(12, p_late))
        demand = int(rng.integers(20, 400))
        rows.append({"part_no": f"P-{i + 1:04d}", "category": cat, "desc": describe(cat, words, spec), **spec,
                     "iface": c["iface"], "country": country, "overseas": overseas, "n_suppliers": n_sup,
                     "lead_time_days": lead, "orders_12m": 12, "late_orders": late,
                     "avg_delay_days": float(round(late and rng.gamma(2, 12 + 20 * p_late), 1)),
                     "eol_status": eol, "ltb_months": int(rng.integers(1, 25)) if eol == "단종예고" else None,
                     "price_change_pct": float(round(rng.normal(4 + 12 * p_late, 8), 1)), "criticality": crit,
                     "annual_demand": demand, "stock_qty": int(rng.integers(0, demand)),
                     "defect_rate": float(round(abs(rng.normal(0.8 + 2 * p_late, 0.8)), 2)),
                     "export_license": bool(overseas and rng.random() < 0.35),
                     "disrupted_next12m": bool(rng.random() < _sig(z + 0.4))})
        if not overseas:
            continue
        # 과거 승인된 국산 대체품(정답): 동의어로 서술, 규격 호환(전력 여유 허용, 일부는 크기 차이가 커 장착 검토 필요)
        eq_words = [c["syn"].get(w, w) for w in words]
        hard = rng.random() < 0.15                                   # 서술이 많이 다른 사례
        if hard:
            eq_words = [c["syn"].get(w, w) for w in c["words"] if w not in words][:1] + eq_words[1:3]
        eq = {**spec, "w": spec["w"] if rng.random() < 0.7 else W_OPTS[min(W_OPTS.index(spec["w"]) + 1, 2)],
              "size_mm": float(round(spec["size_mm"] * (rng.uniform(1.11, 1.15) if rng.random() < 0.12 else rng.uniform(0.95, 1.05)), 1))}
        cat_rows.append({"cand_no": f"D-{len(cat_rows) + 1:04d}", "category": cat, "desc": describe(cat, eq_words, eq), **eq,
                         "iface": c["iface"], "vendor": str(rng.choice(VENDORS)), "approved_for": f"P-{i + 1:04d}", "kind": "equivalent"})
        # 유사 오답 후보: 설명은 거의 같지만 전압이 다르거나(5V/48V) 상용 온도 등급(0~70C)
        for kind in ("volt_mismatch", "temp_mismatch"):
            d = dict(spec)
            if kind == "volt_mismatch":
                d["v"] = int(rng.choice([5, 48]))                           # 체계에서 쓰지 않는 전압
            else:
                d["tmin"], d["tmax"] = 0, 70
            d["size_mm"] = float(round(spec["size_mm"] * rng.uniform(0.95, 1.05), 1))
            cat_rows.append({"cand_no": f"D-{len(cat_rows) + 1:04d}", "category": cat, "desc": describe(cat, words, d), **d,
                             "iface": c["iface"], "vendor": str(rng.choice(VENDORS)), "approved_for": None, "kind": kind})
    parts = pd.DataFrame(rows)
    catalog = pd.DataFrame(cat_rows).sample(frac=1, random_state=seed).reset_index(drop=True)
    return parts, catalog
# ---------------------------------------------------------------------------

# ---- 조달 위험 스크리닝 룰 설정(YAML 형식 문자열: 규정이 바뀌면 코드가 아니라 이 설정만 고친다) ----
RULES_YAML = """
- id: R01
  name: 단일 공급선
  field: n_suppliers
  op: "=="
  value: 1
  weight: 3
- id: R02
  name: 해외 조달 의존
  field: overseas
  op: "=="
  value: true
  weight: 1
- id: R03
  name: 리드타임 180일 초과
  field: lead_time_days
  op: ">"
  value: 180
  weight: 2
- id: R04
  name: 납기 지연률 30% 이상
  field: delay_rate
  op: ">="
  value: 0.3
  weight: 3
- id: R05
  name: 평균 지연 30일 초과
  field: avg_delay_days
  op: ">"
  value: 30
  weight: 2
- id: R06
  name: 단종 예고
  field: eol_status
  op: "=="
  value: 단종예고
  weight: 3
- id: R07
  name: 단종 확정
  field: eol_status
  op: "=="
  value: 단종
  weight: 4
- id: R08
  name: 최종구매 6개월 이내
  field: ltb_months
  op: "<="
  value: 6
  weight: 3
- id: R09
  name: 단가 20% 이상 급등
  field: price_change_pct
  op: ">="
  value: 20
  weight: 2
- id: R10
  name: 핵심(A) 부품 단일 공급
  field: crit_single
  op: "=="
  value: true
  weight: 3
- id: R11
  name: 재고 가용기간 < 리드타임
  field: cover_gap_days
  op: "<"
  value: 0
  weight: 3
- id: R12
  name: 입고 불량률 2% 초과
  field: defect_rate
  op: ">"
  value: 2.0
  weight: 2
- id: R13
  name: 수출 허가 필요
  field: export_license
  op: "=="
  value: true
  weight: 2
- id: R14
  name: 재고 없음
  field: stock_qty
  op: "=="
  value: 0
  weight: 2
- id: R15
  name: 공급국 편중(30% 이상)
  field: country_share
  op: ">="
  value: 0.3
  weight: 1
- id: R16
  name: 지연 이력 있는 단일 공급선
  field: single_late
  op: "=="
  value: true
  weight: 2
"""
GRADE_CUT = {"고위험": 16, "주의": 10}          # 가중치 합 기준 1차 스크리닝 등급


def parse_rules(text):
    """간이 YAML 파서(목록 + key: value). 숫자·true/false·따옴표 문자열을 해석한다."""
    rules, cur = [], None
    for line in text.strip().splitlines():
        if not line.strip():
            continue
        s = line.strip()
        if s.startswith("- "):
            cur = {}; rules.append(cur); s = s[2:]
        k, v = [x.strip() for x in s.split(":", 1)]
        v = v.strip('"')
        if v in ("true", "false"):
            v = v == "true"
        else:
            try:
                v = float(v) if "." in v else int(v)
            except ValueError:
                pass
        cur[k] = v
    return rules


OPS = {"==": lambda a, b: a == b, "!=": lambda a, b: a != b, ">": lambda a, b: a > b, ">=": lambda a, b: a >= b,
       "<": lambda a, b: a < b, "<=": lambda a, b: a <= b}


def derive(df):
    """룰에 쓰는 파생 필드(지연률, 핵심 단일공급, 재고 가용기간 차이, 공급국 비중 등)를 만든다."""
    d = df.copy()
    d["delay_rate"] = d["late_orders"] / d["orders_12m"].clip(lower=1)
    d["crit_single"] = (d["criticality"] == "A") & (d["n_suppliers"] == 1)
    d["cover_gap_days"] = d["stock_qty"] / d["annual_demand"].clip(lower=1) * 365 - d["lead_time_days"]
    d["country_share"] = d["country"].map(d["country"].value_counts(normalize=True)).where(d["overseas"], 0.0)
    d["single_late"] = (d["n_suppliers"] == 1) & (d["late_orders"] >= 3)
    return d


def apply_rules(rec, rules):
    """레코드 1건(dict)에 룰을 적용해 (위험 점수, 걸린 룰 목록)을 돌려준다. 값이 없으면 해당 룰은 건너뛴다."""
    hits = []
    for r in rules:
        v = rec.get(r["field"])
        if v is None or (isinstance(v, float) and np.isnan(v)):
            continue
        if OPS[r["op"]](v, r["value"]):
            hits.append(r)
    score = sum(r["weight"] for r in hits)
    grade = "고위험" if score >= GRADE_CUT["고위험"] else ("주의" if score >= GRADE_CUT["주의"] else "관심")
    return score, grade, [r["id"] for r in hits]

# [교육용가상자료] 룰과 가상 지침 조항의 대응표(실제 규정 조문이 아니라 연결 방식을 보여 주는 예시)
CLAUSE = {"R01": "가상지침 3조1항(공급선 다원화)", "R02": "가상지침 3조2항(해외 의존 관리)", "R03": "가상지침 4조1항(리드타임 관리)",
          "R04": "가상지침 4조2항(납기 준수)", "R05": "가상지침 4조2항(납기 준수)", "R06": "가상지침 5조1항(단종 관리)",
          "R07": "가상지침 5조1항(단종 관리)", "R08": "가상지침 5조2항(최종구매 판단)", "R09": "가상지침 6조1항(가격 변동 점검)",
          "R10": "가상지침 3조3항(핵심 부품 관리)", "R11": "가상지침 4조3항(재고 확보)", "R12": "가상지침 7조1항(품질 관리)",
          "R13": "가상지침 3조4항(수출통제 확인)", "R14": "가상지침 4조3항(재고 확보)", "R15": "가상지침 3조2항(해외 의존 관리)",
          "R16": "가상지침 3조1항(공급선 다원화)"}

rules = parse_rules(RULES_YAML)
parts, _ = make_parts(100, seed=7)
df = derive(parts)
recs = df.to_dict("records")

# 1) 전체 부품 스크리닝과 처리 지연시간(레코드 1건당)
lat = []
res = []
for r in recs:
    t0 = time.perf_counter()
    out = apply_rules(r, rules)
    lat.append((time.perf_counter() - t0) * 1000)
    res.append(out)
df["score"], df["grade"], df["hits"] = [x[0] for x in res], [x[1] for x in res], [x[2] for x in res]
print(f"[룰 설정] 검증 룰 {len(rules)}종(목표 15종 이상: {'달성' if len(rules) >= 15 else '미달'}), 모든 룰에 가상 지침 조항 연결 {all(r['id'] in CLAUSE for r in rules)}")
print(f"룰 처리 지연시간: 평균 {np.mean(lat):.3f}ms, 최대 {np.max(lat):.3f}ms (목표 50ms 이하: {'달성' if max(lat) <= 50 else '미달'})")

# 2) 등급 분포와 과거 단절 이력(다음 12개월 실제 단절 여부) 대비 탐지력
print("\n[1차 스크리닝 등급 분포]")
print(df.groupby("grade").agg(부품수=("part_no", "size"), 실제단절률=("disrupted_next12m", "mean")).reindex(["고위험", "주의", "관심"]).round(2).to_string())
y = df["disrupted_next12m"].to_numpy()
flag = df["grade"].isin(["고위험", "주의"]).to_numpy()
tp = int((flag & y).sum())
print(f"고위험·주의 선별: 실제 단절 부품 {y.sum()}건 중 {tp}건 포착(재현율 {tp / y.sum():.0%}), 선별 {flag.sum()}건 중 정밀도 {tp / max(flag.sum(), 1):.0%}")

# 3) 룰별 적중 건수(어떤 룰이 얼마나 걸리는가)
cnt = pd.Series([h for hs in df["hits"] for h in hs]).value_counts()
tbl = pd.DataFrame([{"룰": r["id"], "이름": r["name"], "가중치": r["weight"], "적중": int(cnt.get(r["id"], 0)), "근거 조항": CLAUSE[r["id"]]} for r in rules])
print("\n[룰별 적중 현황]")
print(tbl.to_string(index=False))

# 4) 상위 위험 부품 5건과 판정 근거(설명 가능성)
print("\n[고위험 상위 5건과 근거]")
for _, r in df.sort_values("score", ascending=False).head(5).iterrows():
    names = ", ".join(next(x["name"] for x in rules if x["id"] == h) for h in r["hits"])
    print(f"  {r['part_no']} {r['category']} 점수 {r['score']:2d} {r['grade']} (적중 룰 {len(r['hits'])}종): {names[:60]}...")

# 5) 규정 변경 대응: 리드타임 기준을 180일에서 150일로 바꾸는 설정 변경(코드 수정 없음)
new_yaml = RULES_YAML.replace("value: 180", "value: 120")
rules2 = parse_rules(new_yaml)
o2 = [apply_rules(r, rules2) for r in recs]
n_score = sum(a != b[0] for a, b in zip(df["score"], o2)); n_grade = sum(a != b[1] for a, b in zip(df["grade"], o2))
print(f"\n[설정 변경] R03 리드타임 기준 180일 -> 120일: 점수가 바뀐 부품 {n_score}건, 등급이 바뀐 부품 {n_grade}건 (코드 수정 없이 YAML만 변경)")
print("처리 후 사람 검토: 스크리닝은 1차 선별이며, 최종 위험 판단과 조치는 조달 담당자가 결정한다.")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].bar(tbl["룰"], tbl["적중"], color="tab:blue"); ax[0].set_title("Rule hits by rule (synthetic)"); ax[0].set_ylabel("parts")
g = df.groupby("grade")["disrupted_next12m"].mean().reindex(["관심", "주의", "고위험"])
ax[1].bar(["low", "watch", "high"], g.values, color=["tab:green", "gold", "tab:red"]); ax[1].set_title("Actual disruption rate by screening grade"); ax[1].set_ylim(0, 1)
plt.tight_layout(); plt.show()

# ---------------------------------------------------------------------------
# [산출물 자동 생성] src/engine/prc07_rule_engine.py
# ---------------------------------------------------------------------------
m03_engine_code = '''"""[PRC07_M03] 16종 조달 위험 스크리닝 비즈니스 룰 엔진 모듈"""
import yaml, time

class ProcurementRuleEngine:
    """YAML 설정 기반 국방 조달 위험 스크리닝 엔진"""
    def __init__(self, config_yaml=None):
        self.rules = []
    
    def evaluate(self, part_data):
        return {"score": 0, "grade": "관심", "triggered_rules": []}
'''
save_deliverable("src/engine/prc07_rule_engine.py", m03_engine_code)


<details>
<summary><b>💡 [참고] 미션 03 예상 실행 결과 (Output)</b></summary>

```text
[룰 설정] 검증 룰 16종(목표 15종 이상: 달성), 모든 룰에 가상 지침 조항 연결 True
룰 처리 지연시간: 평균 0.012ms, 최대 0.043ms (목표 50ms 이하: 달성)

[1차 스크리닝 등급 분포]
       부품수  실제단절률
grade            
고위험     20   0.60
주의      38   0.39
관심      42   0.29
고위험·주의 선별: 실제 단절 부품 39건 중 27건 포착(재현율 69%), 선별 58건 중 정밀도 47%

[룰별 적중 현황]
  룰              이름  가중치  적중               근거 조항
R01          단일 공급선    3  42  가상지침 3조1항(공급선 다원화)
R02        해외 조달 의존    1  85 가상지침 3조2항(해외 의존 관리)
R03    리드타임 180일 초과    2  27  가상지침 4조1항(리드타임 관리)
R04   납기 지연률 30% 이상    3  49    가상지침 4조2항(납기 준수)
R05    평균 지연 30일 초과    2  64    가상지침 4조2항(납기 준수)
R06           단종 예고    3  26    가상지침 5조1항(단종 관리)
R07           단종 확정    4  11    가상지침 5조1항(단종 관리)
R08     최종구매 6개월 이내    3   6  가상지침 5조2항(최종구매 판단)
R09    단가 20% 이상 급등    2   8 가상지침 6조1항(가격 변동 점검)
R10  핵심(A) 부품 단일 공급    3  14 가상지침 3조3항(핵심 부품 관리)
R11  재고 가용기간 < 리드타임    3  40    가상지침 4조3항(재고 확보)
R12    입고 불량률 2% 초과    2  24    가상지침 7조1항(품질 관리)
R13        수출 허가 필요    2  35  가상지침 3조4항(수출통제 확인)
R14           재고 없음    2   1    가상지침 4조3항(재고 확보)
R15  공급국 편중(30% 이상)    1  70 가상지침 3조2항(해외 의존 관리)
R16 지연 이력 있는 단일 공급선    2  35  가상지침 3조1항(공급선 다원화)

[고위험 상위 5건과 근거]
  P-0087 RF증폭기 점수 26 고위험 (적중 룰 11종): 단일 공급선, 해외 조달 의존, 납기 지연률 30% 이상, 평균 지연 30일 초과, 단종 예고, 최종구매 6...
  P-0028 베어링 점수 25 고위험 (적중 룰 11종): 단일 공급선, 해외 조달 의존, 리드타임 180일 초과, 납기 지연률 30% 이상, 평균 지연 30일 초과,...
  P-0066 커넥터 점수 25 고위험 (적중 룰 11종): 단일 공급선, 해외 조달 의존, 리드타임 180일 초과, 납기 지연률 30% 이상, 평균 지연 30일 초과,...
  P-0033 전원모듈 점수 24 고위험 (적중 룰 11종): 단일 공급선, 해외 조달 의존, 리드타임 180일 초과, 납기 지연률 30% 이상, 평균 지연 30일 초과,...
  P-0077 광학렌즈 점수 24 고위험 (적중 룰 10종): 단일 공급선, 해외 조달 의존, 납기 지연률 30% 이상, 평균 지연 30일 초과, 단종 확정, 핵심(A) ...

[설정 변경] R03 리드타임 기준 180일 -> 120일: 점수가 바뀐 부품 27건, 등급이 바뀐 부품 8건 (코드 수정 없이 YAML만 변경)
처리 후 사람 검토: 스크리닝은 1차 선별이며, 최종 위험 판단과 조치는 조달 담당자가 결정한다.
```
</details>
